# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)

**ML Task Type: Ranking + Classification (Hybrid)**

### Primary Task: Ranking
- I need to order pages by urgency/importance for review
- The output is a ranked list (top 50 pages to review first)
- Evaluation uses ranking metrics (precision@K, average precision)

### Secondary Task: Classification
- Each page gets a binary label: "needs review" (positive) vs "doesn't need review" (negative)
- The model predicts a probability score (0-1) for each page
- Higher probability = higher rank in the review queue

### Why This Hybrid Approach
1. **Ranking** matches the real workflow: reviewers have limited capacity and need pages sorted by priority
2. **Classification** provides the probability scores that drive the ranking
3. **Combined**: We get both the priority order AND an interpretable score

### Why Not Other Task Types

| Task Type | Why It Doesn't Fit |
|-----------|-------------------|
| Pure Regression | Predicting exact traffic change is too noisy - there's too much randomness |
| Pure Classification | Yes/no is too binary; we need to know which pages to look at FIRST |
| Clustering | We want to rank pages by urgency, not group similar pages together |
| Pure Ranking (no scores) | Need interpretable scores so reviewers understand why a page was flagged |

### Real-World Analogy
**Emergency Room Triage**:
- Many patients (pages) arrive at once
- Limited doctor capacity (reviewer time)
- Need to rank patients by urgency (score/priority)
- Treat the most urgent first (top-K)
- Can't treat everyone immediately (capacity constraint)
- A simple rule ("treat everyone with fever") would miss serious cases with no fever

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

### The Ideal Target
> "Will this page decline in the next 30 days?"

This is what we ultimately want to predict - future traffic decline so we can act before it happens.

### The Proxy Target (Week 1-3 Starter)
> `trend_direction == "down"`

This is the starter dataset's label: is the page currently declining in the observed window?

**Why Use a Proxy First?**
1. **Simple**: Already calculated in the data - no extra work needed
2. **Immediate**: No need to build complex time windows
3. **Baseline**: Good for initial experiments to prove the approach works
4. **Comparable**: Matches the starter pipeline results (precision@50 = 74%)

### The Final Target (Week 4-7)
> Future decline over next 30 days

Using the daily warehouse data, I'll build:
- **Feature window**: Last 90 days of performance data
- **Target window**: Next 30 days after the feature window
- **Label**: Was there a significant decline?

### Label Definition

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

### Primary Metric: Precision@K

**Precision@K** measures: out of the top K pages we recommend, how many actually need attention?

### Why Precision@K?
1. **Matches real workflow**: Reviewers check the top N pages each week
2. **Business-relevant**: False positives waste reviewer time, false negatives miss declining pages
3. **Interpretable**: "Out of 50 pages we flagged, 37 were actually declining"
4. **Actionable**: If precision is low, reviewers won't trust the system

### Target Values from Starter Data

| Method | Precision@50 | Interpretation |
|--------|-------------|----------------|
| Baseline (simple rules) | 24% | 12 right out of 50 pages |
| Random Forest (starter) | 74% | 37 right out of 50 pages |
| **My Goal** | **≥75%** | Beat the starter model |

### Secondary Metrics

| Metric | Why It Matters |
|--------|----------------|
| **Average Precision (AP)** | Measures entire ranking quality, not just top-K |
| **Recall@K** | How many of ALL declining pages did we catch in the top-K? |
| **AUC-ROC** | How well does the model separate declining vs non-declining? |
| **False Positive Rate** | How often do we waste reviewer time on pages that are fine? |

### Business Translation


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# 4. The unit of analysis, as a real dataframe
# Creating sample data that shows what the actual data would look like

import pandas as pd
import numpy as np

print("=" * 60)
print("UNIT OF ANALYSIS: ONE PAGE = ONE ROW")
print("=" * 60)

# Create sample data showing what the starter data would contain
np.random.seed(42)
n_pages = 15

sample_data = {
    'page_id': [f'page_{i:03d}' for i in range(1, n_pages + 1)],
    'impressions_90d': np.random.randint(100, 5000, n_pages),
    'clicks_90d': np.random.randint(10, 500, n_pages),
    'ctr': np.random.uniform(0.02, 0.12, n_pages),
    'avg_position': np.random.uniform(1.5, 18.0, n_pages),
    'sessions_90d': np.random.randint(30, 1000, n_pages),
    'content_age_days': np.random.randint(30, 500, n_pages),
    'word_count': np.random.randint(400, 3500, n_pages),
    'trend_direction': np.random.choice(['up', 'down', 'flat'], n_pages, p=[0.35, 0.40, 0.25]),
    'needs_review': np.random.choice([0, 1], n_pages, p=[0.6, 0.4])
}

# Make the data more realistic - high position = lower CTR
for i in range(n_pages):
    if sample_data['avg_position'][i] > 10:
        sample_data['ctr'][i] = np.random.uniform(0.01, 0.04)
    else:
        sample_data['ctr'][i] = np.random.uniform(0.03, 0.12)

df = pd.DataFrame(sample_data)

print("\n📊 Sample Data (15 pages - showing the unit of analysis)")
print("-" * 60)
print("EACH ROW = ONE PAGE (this is the unit of analysis)")
print("-" * 60)
print(df.to_string(index=False))
print("-" * 60)

print(f"\nTotal pages in this sample: {len(df)}")

print("\n" + "=" * 60)
print("WHAT EACH ROW REPRESENTS")
print("=" * 60)
print("Unit: One content page (one URL)")
print("Features: 90 days of performance + content metadata")
print("Target: Does this page need review? (1 = yes, 0 = no)")

print("\n" + "=" * 60)
print("FEATURE BREAKDOWN BY TYPE")
print("=" * 60)
print("\n🔍 SEARCH SIGNALS:")
print("  - impressions_90d: How often this page appears in search")
print("  - clicks_90d: How many times people clicked through")
print("  - ctr: Click-through rate (clicks ÷ impressions)")
print("  - avg_position: Average search result position (lower = better)")

print("\n📊 ENGAGEMENT SIGNALS:")
print("  - sessions_90d: How many visits to this page")
print("  - content_age_days: How old is this content (days since published)")

print("\n📝 CONTENT SIGNALS:")
print("  - word_count: How long is the content")
print("  - trend_direction: Traffic trend (up/down/flat)")

print("\n🎯 TARGET (what we predict):")
print("  - needs_review: Should this page be in this week's review queue?")

print("\n" + "=" * 60)
print("WHY THIS UNIT MAKES SENSE FOR THE PROBLEM")
print("=" * 60)
print("1. ✅ Pages are the natural unit of action (a reviewer edits ONE page)")
print("2. ✅ Pages have all the signals we care about (traffic, engagement, content)")
print("3. ✅ Pages are what appear in the final ranked list (top 50 pages)")
print("4. ✅ Pages match the reviewer's workflow (they open and evaluate one page at a time)")
print("5. ✅ Pages have clear decisions (refresh, rewrite, merge, or monitor)")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Unit: Page")
print(f"Rows: {len(df)} pages in this sample")
print(f"Columns: {len(df.columns)} features")
print(f"Goal: Rank pages by need for review (highest priority first)")


UNIT OF ANALYSIS: ONE PAGE = ONE ROW

📊 Sample Data (15 pages - showing the unit of analysis)
------------------------------------------------------------
EACH ROW = ONE PAGE (this is the unit of analysis)
------------------------------------------------------------
 page_id  impressions_90d  clicks_90d      ctr  avg_position  sessions_90d  content_age_days  word_count trend_direction  needs_review
page_001              960         323 0.083898      3.513631           845               439        2427            flat             0
page_002             3872          31 0.092531      9.670419           300               246        3095            flat             0
page_003             3192         262 0.109242      2.067411           485               281        1895            down             1
page_004              566         245 0.028731     16.503787           491               217        1562              up             0
page_005             4526         354 0.056607      5.7698

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

### The Problem with Fixed Rules

**Simple rules** (like the starter baseline) use hard thresholds:

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.